<a href="https://colab.research.google.com/github/junseo25/ds2002-fa26/blob/main/notebooks/04-json-apis/2026_10_02_Jun_Seo_Lee_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [13]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [14]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id'],
)

# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [15]:
# TODO
joined = pd.merge(
    lines,
    pd.json_normalize(vendors_json),
    on='vendor_id',
    how='left',
    indicator=True,
)

unmatched = joined[joined['_merge'] == 'left_only']

print('unmatched vendor:', unmatched['vendor_id'].unique().tolist())
print('units attached to unmatched vendor:', unmatched['qty'].sum())

joined

unmatched vendor: ['V-07']
units attached to unmatched vendor: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [16]:
# TODO
joined['name'] = joined['name'].fillna('Unknown vendor')
joined.groupby('name')['qty'].sum()

# Keep the unmatched seller as unknown vendor instead of losing the data entirely.

,qty
name,
Cav Merch,1
Hoos Burgers,3
Unknown vendor,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [17]:
# TODO
joined['zone'] = joined['zone'].fillna('Unknown zone')
joined.groupby('zone')['qty'].sum()

#Total: 7 with the ones that were unmatched going into unknown zone

,qty
zone,
A,4
Unknown zone,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [18]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
records_with_lines = [record for record in ragged_json if 'lines' in record]
ragged_lines = pd.json_normalize(
    records_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id'],
    errors='ignore',
)

# The missing Fries quantity stays NaN, which remains different from a real zero.
print(ragged_lines)
print('missing quantities:', ragged_lines['qty'].isna().sum())

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01
missing quantities: 1


### Q6 — Validate

In [19]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per item gives each item its own item and quantity while still allowing for order and vendor IDs to be retained

b) A missing quantity will mean the number of units will be unknown and unreported while a zero is that the quantity is a known value at zero. Downstream, it could appear as though the information is complete when it is actually understating the true sales values. Preserving the information would allow for true knowledge of the sales and the ability to fix unknown values later as the problem becomes known.